# Dataset Verification and Overview

**Project:** Context-Aware Trust Scoring and Review-Based Product Recommendation  
**Dataset:** Amazon Reviews 2018 (Fashion Category)

**Objective:**  
To verify that the dataset is usable, well-structured, and suitable for trust scoring and recommendation tasks.


Import Required Libraries

In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_colwidth', 300)
pd.set_option('display.max_columns', None)


Load Dataset

In [2]:
import os
import gzip
import json
import shutil
import requests
from pathlib import Path

# Define paths relative to the project root
PROJECT_ROOT = Path("..").resolve()
DATA_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PATH = DATA_DIR / "AMAZON_FASHION.json"
DATA_URL = "http://deepyeti.ucsd.edu/jianmo/amazon/categoryFiles/AMAZON_FASHION.json.gz"

# Ensure directory exists
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Download and unzip if file doesn't exist
if not DATA_PATH.exists():
    print(f"File not found at {DATA_PATH}. Downloading from {DATA_URL}...")
    try:
        response = requests.get(DATA_URL, stream=True)
        response.raise_for_status()
        
        # Download compressed file
        compressed_path = DATA_PATH.with_suffix(".json.gz")
        with open(compressed_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                f.write(chunk)
        
        print("Download complete. Extracting...")
        
        # Extract json.gz to json
        with gzip.open(compressed_path, 'rb') as f_in:
            with open(DATA_PATH, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
        
        # Clean up compressed file
        compressed_path.unlink()
        print("Extraction complete.")
        
    except Exception as e:
        print(f"Error downloading/extracting dataset: {e}")
        raise

print(f"Loading dataset from: {DATA_PATH}")
df = pd.read_json(DATA_PATH, lines=True)
print("Dataset loaded successfully!")


Loading dataset from: D:\Context_Aware_Trust_Scoring_Recommendation_Fashion\data\raw\AMAZON_FASHION.json
Dataset loaded successfully!


In [3]:
print(f"Total number of reviews: {df.shape[0]}")


Total number of reviews: 883636


In [4]:
print("Column names:\n")
print(df.columns.tolist())


Column names:

['overall', 'verified', 'reviewTime', 'reviewerID', 'asin', 'reviewerName', 'reviewText', 'summary', 'unixReviewTime', 'vote', 'style', 'image']


In [5]:
df.dtypes


overall             int64
verified             bool
reviewTime         object
reviewerID         object
asin               object
reviewerName       object
reviewText         object
summary            object
unixReviewTime      int64
vote              float64
style              object
image              object
dtype: object

In [6]:
df.sample(5)[
    ["reviewerID", "asin", "overall", "reviewText", "summary", "verified"]
]


,reviewerID,asin,overall,reviewText,summary,verified
398423,AFQ6W1CC6NK16,B0009A6KFA,5,This ring was perfect and not to extravagant. Not too simple but just right. My fiance loves it. Could not find anything this simple and still awesome in stores. Fiance is happy and I am happy too!,Beautiful ring,True
22654,A2YC3I62L74I2J,B000PHANNM,5,IPlay makes great products! My twins wear their hats well? The fabric is a quality thin fabric that stays cool on their heads.,Perfect!,True
772755,A3LMPSHVCGWVUQ,B01A6WF5OC,3,"Its a long sleeve dress, not over the shoulder! Still looks cute though! Shipped from China, so don't expect high quality.","Its a long sleeve dress, not over the shoulder ...",True
396196,A3711353Y8BHSA,B01HBLM8EQ,5,"This is my first order with this company and I couldn't be more pleased. The quality of the product is top-notch. The fabric is sturdy, yet soft, and will definitely hold up to outdoor patio use. The pattern is true to the picture, and the colors are more vibrant than they appear in the picture ...","Great product, great company",False
694542,A19U30ODI5R9SP,B0148UA0CK,5,"This is my most fav waist trainer. I wear daily. It gives me a nice shape and is cute enough to where if your shirt rides up, it won't matter.lol im a mediumvand it fit great. It also is perfect for when you do wraps on your tummy. It has the latex lining that makes your belly sweat and i swear ...",It gives me a nice shape and is cute enough to where if your ...,True


In [7]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage.sort_values(ascending=False)


image             96.739947
vote              90.957815
style             65.532301
reviewText         0.139537
summary            0.060319
reviewerName       0.010412
verified           0.000000
overall            0.000000
reviewerID         0.000000
reviewTime         0.000000
asin               0.000000
unixReviewTime     0.000000
dtype: float64

In [8]:
# Reviews per user
reviews_per_user = df["reviewerID"].value_counts()
print("Users with only 1 review:", (reviews_per_user == 1).sum())

# Reviews per product
reviews_per_product = df["asin"].value_counts()
print("Products with only 1 review:", (reviews_per_product == 1).sum())

Users with only 1 review: 655320
Products with only 1 review: 99962


## Dataset Summary

- The Amazon Fashion dataset contains user reviews with textual feedback and ratings.
- Each review is associated with a unique user (`reviewerID`) and product (`asin`).
- Ratings are numeric, and reviews include rich free-text content.
- Additional metadata such as verification status and timestamps are available.
- The dataset is suitable for:
  - Review-level trust scoring
  - Product-level recommendation aggregation
